# 📞 Conseiller Bancaire BANCOBU - Mode Appel Téléphonique IA

Ce projet transforme votre assistant bancaire en un **véritable conseiller téléphonique IA** en temps réel :
1. 🎙️ **Écoute de l'appelant (STT) :** Transcription vocale ultra-rapide avec **Whisper** (`openai/whisper-small` accéléré sur GPU T4).
2. 🔍 **Recherche documentaire (RAG) :** Analyse instantanée des fiches bancaires BANCOBU dans **Qdrant**.
3. 🧠 **Intelligence & Élocution :** **Google Gemini** formate des réponses orales, chaleureuses et professionnelles adaptées à un appel téléphonique.
4. 🔊 **Réponse vocale (TTS) :** Synthèse vocale neuronale réaliste (**Edge-TTS**) avec voix française naturelle (Conseillère Denise ou Conseiller Henri).
5. 📱 **Interface Appel Web :** Déploiement en un clic avec **Gradio** et lien public de partage.

---
⚡ **Rappel Colab :** Activez le GPU gratuit dans `Exécution` > `Modifier le type d'exécution` > Choisir **T4 GPU**.

### 1. Installation des dépendances nécessaires

In [ ]:
# Installation de toutes les briques nécessaires (avec fix PyAV av<18, edge-tts et librairies CUDA 12)
!pip install -q pymupdf qdrant-client google-genai faster-whisper gradio sentence-transformers edge-tts "av<18" nvidia-cublas-cu12 nvidia-cudnn-cu12


### 2. Configuration & Clé API Gemini

In [ ]:
import os
import getpass
from google import genai

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = None

if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass.getpass("Entrez votre clé API Gemini (Google AI Studio) : ")

GEMINI_MODEL = "gemini-2.5-flash"
gemini_client = genai.Client(api_key=GEMINI_API_KEY)
print("✅ Connexion à Gemini établie avec succès !")

### 3. Téléversement du document PDF bancaire

In [ ]:
import os
from google.colab import files

pdf_filename = "bancobu_donnees.pdf"

if not os.path.exists(pdf_filename):
    print("Veuillez sélectionner le fichier bancobu_donnees.pdf à téléverser :")
    uploaded = files.upload()
    for fname in uploaded.keys():
        if fname.endswith(".pdf"):
            pdf_filename = fname
            break

print(f"✅ Fichier prêt pour l'indexation : {pdf_filename}")

### 4. Extraction du PDF et Indexation Vectorielle dans Qdrant

In [ ]:
import re
import hashlib
import pymupdf
import torch
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models

def extract_pages(pdf_path):
    doc = pymupdf.open(pdf_path)
    pages = []
    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text")
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text).strip()
        if text:
            pages.append({"page": page_num, "text": text})
    doc.close()
    return pages

CHUNK_SIZE = 1500
CHUNK_OVERLAP = 200

def split_text(text):
    paragraphs = [p.strip() for p in text.split("\n") if p.strip()]
    chunks = []
    current = ""
    for p in paragraphs:
        candidate = f"{current}\n{p}" if current else p
        if len(candidate) <= CHUNK_SIZE:
            current = candidate
        else:
            if current:
                chunks.append(current)
            overlap = current[-CHUNK_OVERLAP:] if current else ""
            current = f"{overlap}\n{p}".strip()
    if current:
        chunks.append(current)
    return chunks

pages = extract_pages(pdf_filename)
documents = []
for p in pages:
    for idx, chunk in enumerate(split_text(p["text"])):
        documents.append({
            "text": chunk,
            "page": p["page"],
            "chunk": idx,
            "source": pdf_filename
        })

print(f"📄 {len(pages)} pages extraites, {len(documents)} chunks créés.")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚙️ Chargement du modèle d'embedding sur {device.upper()}...")
embed_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=device)

qdrant = QdrantClient(":memory:")
COLLECTION_NAME = "bancobu_docs"
vector_size = embed_model.get_sentence_embedding_dimension()

qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(size=vector_size, distance=models.Distance.COSINE)
)

print("📥 Encodage des documents et indexation...")
texts = [doc["text"] for doc in documents]
embeddings = embed_model.encode(texts, batch_size=32, show_progress_bar=True)

points = []
for i, (doc, vector) in enumerate(zip(documents, embeddings)):
    points.append(models.PointStruct(
        id=i,
        vector=vector.tolist(),
        payload=doc
    ))

qdrant.upsert(collection_name=COLLECTION_NAME, points=points)
print(f"✅ Base vectorielle prête ! ({len(points)} extraits indexés)")

### 5. Reconnaissance Vocale (Whisper) & Synthèse Vocale Téléphonique (Edge-TTS)

In [ ]:
import av
import re
from faster_whisper import WhisperModel
import edge_tts

# 1. Patch de compatibilité PyAV >= 18
_orig_av_open = av.open
def _safe_av_open(*args, **kwargs):
    kwargs.pop("metadata_errors", None)
    return _orig_av_open(*args, **kwargs)
av.open = _safe_av_open

# 2. Chargement de Whisper en mode CPU int8
# Sur Python 3.13, le mode CPU int8 est ultra rapide (1 seconde), léger, et évite à 100% les erreurs libcublas.so
print("🎙️ Chargement de Whisper (CPU optimisé int8)...")
whisper_model = WhisperModel("small", device="cpu", compute_type="int8")

def transcribe(audio_path):
    if not audio_path:
        return ""
    initial_prompt = "BANCOBU, compte bancaire, compte épargne, devises, virement, crédit, microfinance."
    segments, info = whisper_model.transcribe(
        audio_path,
        language="fr",
        initial_prompt=initial_prompt,
        beam_size=5
    )
    text_parts = [seg.text for seg in segments]
    return " ".join(text_parts).strip()

# 3. Synthèse Vocale Téléphonique (Text-to-Speech)
async def generate_speech(text, voice="fr-FR-DeniseNeural", output_file="reponse_conseiller.mp3"):
    clean_text = re.sub(r"[*_#>`~]", "", text)
    clean_text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", clean_text)
    clean_text = clean_text.replace("\n", " ").strip()
    
    communicate = edge_tts.Communicate(clean_text, voice=voice, rate="+0%", pitch="+0Hz")
    await communicate.save(output_file)
    return output_file

print("✅ Moteurs de voix (Whisper & Synthèse Téléphonique) prêts !")


### 6. Logique Conseiller Bancaire (Conversation Téléphonique & RAG)

In [ ]:
def retrieve_passages(query, top_k=5):
    query_vector = embed_model.encode(query).tolist()
    results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
        with_payload=True
    )
    return results.points

def generate_phone_answer(question, passages):
    if not passages:
        return "Bonjour ! Je ne trouve malheureusement pas cette information dans les documents disponibles de la banque. N'hésitez pas à contacter votre agence BANCOBU la plus proche."

    context = ""
    for idx, point in enumerate(passages, start=1):
        data = point.payload
        extrait = data.get("text", "")
        context += f"EXTRAIT {idx} :\n{extrait}\n\n"

    # Prompt spécifiquement conçu pour un appel vocal
    prompt = f"""
Tu es un conseiller clientèle téléphonique professionnel et bienveillant de la banque BANCOBU.
Un client est en ligne au téléphone avec toi et te pose une question.

CONSIGNES POUR L'APPEL TÉLÉPHONIQUE :
1. Adopte un ton oral naturel, fluide, chaleureux et rassurant.
2. Fais des phrases claires et agréables à écouter. Évite impérativement le formatage écrit : PAS de listes à puces avec astérisques (*), PAS de titres (#), PAS de tableaux.
3. Reste concis (environ 3 à 5 phrases) : un client au téléphone souhaite une réponse directe et précise sans monologue interminable.
4. Base-toi EXCLUSIVEMENT sur les extraits documentaires fournis ci-dessous. N'invente aucune condition ni tarif.
5. Si l'information est introuvable ou incomplète, indique-le aimablement au client.

QUESTION DU CLIENT AU TÉLÉPHONE :
{question}

EXTRAITS DOCUMENTAIRES BANCOBU DISPONIBLES :
{context}

Donne maintenant ta réponse orale au client :
"""

    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt
    )
    return response.text.strip()

async def handle_call(audio_input, text_input, voice_choice):
    question = ""
    if audio_input:
        question = transcribe(audio_input)
    elif text_input:
        question = text_input.strip()
        
    if not question:
        msg = "Veuillez poser votre question à la voix (microphone) ou par écrit."
        return msg, "", None

    # 1. Recherche RAG
    passages = retrieve_passages(question)
    
    # 2. Réponse orale du conseiller
    answer = generate_phone_answer(question, passages)
    
    # 3. Synthèse vocale de la réponse
    voice_name = "fr-FR-DeniseNeural" if "Denise" in voice_choice else "fr-FR-HenriNeural"
    audio_path = await generate_speech(answer, voice=voice_name)
    
    # 4. Renvoi des 3 valeurs attendues par Gradio : (transcription, texte, fichier audio)
    return question, answer, audio_path


### 7. Interface Web Interactive (Simulateur d'Appel Téléphonique avec Gradio)

In [ ]:
import gradio as gr

with gr.Blocks(theme=gr.themes.Soft(), title="Appel Conseiller Bancaire BANCOBU") as demo:
    gr.Markdown("""
    # ☎️ BANCOBU - Service Client Téléphonique
    Appuyez sur le micro pour parler à votre conseiller bancaire, ou écrivez votre question.
    L'assistant vous répondra directement à la voix !
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 👤 Votre Appel")
            audio_input = gr.Audio(
                sources=["microphone", "upload"],
                type="filepath",
                label="🎙️ Parler dans le combiné (Microphone)"
            )
            text_input = gr.Textbox(
                label="Ou saisir votre question au clavier :",
                placeholder="Ex: Quelles sont les conditions pour ouvrir un compte épargne ?",
                lines=2
            )
            voice_choice = gr.Radio(
                choices=["Conseillère (Denise)", "Conseiller (Henri)"],
                value="Conseillère (Denise)",
                label="Choix du conseiller vocal :"
            )
            call_btn = gr.Button("📞 Lancer la communication", variant="primary", size="lg")
            clear_btn = gr.ClearButton([audio_input, text_input])
            
        with gr.Column(scale=1):
            gr.Markdown("### 🏦 En direct avec BANCOBU")
            recognized_box = gr.Textbox(label="📝 Ce que le conseiller a entendu :", interactive=False)
            audio_output = gr.Audio(
                label="🔊 Voix du Conseiller (Réponse audio) :",
                type="filepath",
                autoplay=True
            )
            transcript_box = gr.Markdown(label="📄 Transcription de la réponse :")

    call_btn.click(
        fn=handle_call,
        inputs=[audio_input, text_input, voice_choice],
        outputs=[recognized_box, transcript_box, audio_output]
    )

demo.launch(share=True, debug=True)